# vR.P.17 --- Image Tampering Detection and Localization
## ELA + DCT Spatial Fusion (6-Channel Input)

---

| Field | Value |
|-------|-------|
| **Version** | vR.P.17 |
| **Track** | Pretrained Localization (Track 2) |
| **Architecture** | UNet with ResNet-34 encoder (ImageNet pretrained, **frozen body + BN unfrozen + conv1 unfrozen**) |
| **Change** | Replace 3ch ELA with 6ch ELA+DCT fusion input |
| **Parent** | vR.P.3 (ELA input Q=90, frozen body + BN unfrozen) |
| **Dataset** | CASIA v2.0 --- Splicing Detection + Localization (~12,614 images + GT masks) |
| **Platform** | Kaggle (T4/P100 GPU) |
| **Framework** | PyTorch + Segmentation Models PyTorch (SMP) |

---

### Pipeline Overview

```
Raw RGB Image
 ├── ELA (Q=90) -> 3-channel ELA map (384x384)
 └── DCT -> 3-channel spatial DCT map (48x48 -> upsampled to 384x384)
         |
         v
    Concatenate -> 6-channel input (384x384)
         |
         v
    UNet (ResNet-34 encoder, conv1 modified for 6ch)
    - conv1 UNFROZEN (learns 6-channel input)
    - body FROZEN + BN UNFROZEN
    - decoder TRAINABLE
         |
         v
    384x384 binary mask -> Pixel F1, IoU, AUC + Image-level accuracy
```

### What Changed from vR.P.3

- **Input:** 3ch ELA (Q=90) -> **6ch ELA (Q=90) + DCT spatial map**
- **conv1:** Frozen (pretrained 3ch) -> **Unfrozen (modified for 6ch, pretrained weights duplicated and scaled by 0.5)**
- **Normalization:** Single ELA mean/std -> **Separate ELA + DCT mean/std**

ELA captures pixel-level recompression artifacts (spatial domain). DCT captures block-level compression statistics (frequency domain). Fusing both provides the model with two independent views of tampering evidence.

---

## Change Log

| Version | Track | Change | Result |
|---------|-------|--------|--------|
| vR.1.0 | ETASR | Paper reproduction baseline | 89.89% val acc |
| vR.1.1 | ETASR | Proper test split + metrics | 88.38% test acc, AUC=0.9601 |
| vR.P.0 | Pretrained | ResNet-34 UNet, frozen encoder, RGB input | Baseline |
| vR.P.1 | Pretrained | Fix dataset discovery + GT mask detection | Dataset fix |
| vR.P.2 | Pretrained | Gradual unfreeze: layer3+layer4, differential LR | Unfreeze |
| vR.P.3 | Pretrained | ELA as input Q=90 (replace RGB), frozen+BN | Pixel F1 = 0.6920 |
| vR.P.10 | Pretrained | CBAM attention + Focal+Dice | Pixel F1 = 0.7277 |
| vR.P.15 | Pretrained | Multi-quality ELA (Q=75/85/95 as 3 channels) | Pending |
| vR.P.16 | Pretrained | DCT spatial feature maps (AC/DC/HF as 3 channels) | Pending |
| **vR.P.17** | **Pretrained** | **ELA + DCT fusion (6-channel input)** | **This notebook** |

### What Changed from vR.P.3

| Aspect | vR.P.3 | vR.P.17 (This Version) |
|--------|--------|------------------------|
| Input | 3ch ELA (Q=90) | **6ch ELA (Q=90) + DCT spatial map** |
| IN_CHANNELS | 3 | **6** |
| conv1 | Frozen (pretrained) | **Unfrozen (modified for 6ch)** |
| Normalization | ELA mean/std only | **Separate ELA + DCT mean/std** |

### What DID NOT change
- Architecture: UNet + ResNet-34
- Encoder state: Frozen body + BN unfrozen (conv1 now unfrozen)
- Loss: BCEDiceLoss
- Optimizer: Adam, single LR=1e-3
- Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)
- Epochs: 25, Patience: 7
- Data split: 70/15/15 stratified
- Seed: 42

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

# Install segmentation-models-pytorch
!pip install -q segmentation-models-pytorch

import os
import sys
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageChops, ImageEnhance
from io import BytesIO
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# ============================================================
# Configuration
# ============================================================
VERSION = 'vR.P.17'
CHANGE = 'ELA + DCT fusion (6-channel input)'
SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 16
ENCODER = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
IN_CHANNELS = 6  # 3 ELA + 3 DCT
NUM_CLASSES = 1
LEARNING_RATE = 1e-3  # Single LR (decoder + BN + conv1)
INPUT_TYPE = 'ELA+DCT'
ELA_QUALITY = 90
DCT_BLOCK_SIZE = 8
EPOCHS = 25
PATIENCE = 7
NUM_WORKERS = 2  # Parallel data loading
CHECKPOINT_PATH = f'{VERSION}_checkpoint.pth'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    # TF32 for faster math on Ampere+ GPUs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# GPU info
print(f"{'='*60}")
print(f"  {VERSION} — {CHANGE}")
print(f"{'='*60}")
print(f"Device:   {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch:  {torch.__version__}")
print(f"SMP:      {smp.__version__}")
print(f"Image:    {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Input:    ELA (Q={ELA_QUALITY}) + DCT spatial maps (6ch fusion)")
print(f"Encoder:  {ENCODER} ({ENCODER_WEIGHTS}, frozen body + BN unfrozen + conv1 unfrozen)")
print(f"Batch:    {BATCH_SIZE}")
print(f"LR:       {LEARNING_RATE} (decoder + BN + conv1)")
print(f"Epochs:   {EPOCHS}")
print(f"Patience: {PATIENCE}")
print(f"Seed:     {SEED}")
print(f"Workers:  {NUM_WORKERS}")
print(f"AMP:      Enabled (mixed precision)")
print(f"TF32:     Enabled")

---

## 2. Dataset

### CASIA v2.0

The CASIA v2.0 dataset contains ~12,614 images:
- **Authentic (Au):** ~7,491 images â€” unmodified photographs
- **Tampered (Tp):** ~5,123 images â€” images with splicing or copy-move forgery

For **localization**, we need pixel-level ground truth masks indicating which regions are tampered. CASIA v2.0 includes ground truth masks for tampered images. If GT masks are not available in the Kaggle dataset, we generate pseudo-masks using ELA thresholding as a fallback.

For **authentic images**, the ground truth mask is all zeros (no tampering).

In [ ]:
# ============================================================
# 2.1 Dataset Path Discovery (FIXED in vR.P.1)
# ============================================================

def find_dataset():
    """Search /kaggle/input/ for Au/ and Tp/ directories.
    
    FIXED in vR.P.1: Collects ALL candidate dirs containing Au+Tp,
    then prefers the one with 'image' in the path (not 'mask').
    Also detects sibling MASK directory as ground truth.
    
    Returns: (dataset_root, au_dir, tp_dir, gt_mask_dir_or_None)
    """
    search_roots = ['/kaggle/input', '/content/drive/MyDrive']
    candidates = []  # list of (dirpath, au_path, tp_path)
    
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if 'Au' in dirnames and 'Tp' in dirnames:
                candidates.append((
                    dirpath,
                    os.path.join(dirpath, 'Au'),
                    os.path.join(dirpath, 'Tp')
                ))
    
    if not candidates:
        return None, None, None, None
    
    # Separate IMAGE vs MASK candidates
    image_candidates = [c for c in candidates if 'mask' not in c[0].lower()]
    mask_candidates = [c for c in candidates if 'mask' in c[0].lower()]
    
    # Prefer IMAGE directory; fall back to first candidate if no IMAGE found
    if image_candidates:
        # Among image candidates, prefer the one with 'image' in path
        explicit_image = [c for c in image_candidates if 'image' in c[0].lower()]
        chosen = explicit_image[0] if explicit_image else image_candidates[0]
    else:
        chosen = candidates[0]
    
    # Detect GT mask directory
    gt_dir = None
    if mask_candidates:
        # Use the MASK candidate that has Au/Tp structure (for tampered masks)
        gt_dir = mask_candidates[0][0]  # root of MASK/{Au,Tp}
    
    return chosen[0], chosen[1], chosen[2], gt_dir

DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()

if DATASET_ROOT is None:
    for base in ['/kaggle/input']:
        if os.path.isdir(base):
            print(f'Contents of {base}:')
            for dirpath, dirnames, filenames in os.walk(base):
                depth = dirpath.replace(base, '').count(os.sep)
                print(f'{"  " * depth}{os.path.basename(dirpath)}/')
                if depth >= 3:
                    break
    raise FileNotFoundError('Could not find Au/ and Tp/ directories.')

# Resolve GT mask directory
# GT_DIR_ROOT may be MASK/ which contains Au/ and Tp/ subdirs.
# For tampered images, GT masks are in MASK/Tp/
# We need to check if MASK/Tp/ contains actual mask images.
GT_DIR = None
if GT_DIR_ROOT is not None:
    # Check if MASK/Tp/ has image files (those are GT masks for tampered images)
    gt_tp_dir = os.path.join(GT_DIR_ROOT, 'Tp')
    if os.path.isdir(gt_tp_dir):
        gt_files = [f for f in os.listdir(gt_tp_dir)
                    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.bmp'}]
        if gt_files:
            GT_DIR = GT_DIR_ROOT  # MASK directory with Au/Tp structure
            print(f'GT mask structure detected: {GT_DIR}')
            print(f'  MASK/Au: {len(os.listdir(os.path.join(GT_DIR, "Au")))} files')
            print(f'  MASK/Tp: {len(gt_files)} mask files')

# If GT_DIR_ROOT didn't work, search for other GT mask directories
if GT_DIR is None:
    # Search within dataset neighborhood
    search_base = os.path.dirname(DATASET_ROOT)
    for root, dirs, files in os.walk(search_base):
        for d in dirs:
            if any(pat in d.lower() for pat in ['groundtruth', 'gt', 'mask']):
                candidate = os.path.join(root, d)
                if any(Path(f).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'}
                       for f in os.listdir(candidate) if os.path.isfile(os.path.join(candidate, f))):
                    GT_DIR = candidate
                    break
        if GT_DIR:
            break
    
    # Search separate Kaggle datasets
    if GT_DIR is None:
        input_dir = '/kaggle/input'
        if os.path.isdir(input_dir):
            for d in sorted(os.listdir(input_dir)):
                if any(pat in d.lower() for pat in ['groundtruth', 'gt', 'mask']):
                    for root, dirs, files in os.walk(os.path.join(input_dir, d)):
                        img_files = [f for f in files if Path(f).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'}]
                        if img_files:
                            GT_DIR = root
                            break
                    if GT_DIR:
                        break

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

print(f'\nDataset root:  {DATASET_ROOT}')
print(f'Authentic dir: {AU_DIR}  ({len(os.listdir(AU_DIR))} files)')
print(f'Tampered dir:  {TP_DIR}  ({len(os.listdir(TP_DIR))} files)')
if GT_DIR:
    print(f'GT mask dir:   {GT_DIR}')
else:
    print(f'GT mask dir:   NOT FOUND \u2014 will generate pseudo-masks from ELA')


In [ ]:
# ============================================================
# 2.2 Collect Image Paths and Ground Truth Masks (UPDATED)
# ============================================================

def collect_paths(directory):
    """Collect sorted image paths from a directory."""
    paths = []
    for f in sorted(os.listdir(directory)):
        if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
            paths.append(os.path.join(directory, f))
    return paths

au_paths = collect_paths(AU_DIR)
tp_paths = collect_paths(TP_DIR)

# Build GT mask lookup
# The GT directory may have MASK/Au/ and MASK/Tp/ structure
# OR it may be a flat directory with mask files
gt_map = {}
if GT_DIR:
    gt_tp_dir = os.path.join(GT_DIR, 'Tp')
    gt_au_dir = os.path.join(GT_DIR, 'Au')
    
    # If GT has Au/Tp structure, collect from Tp subdirectory
    if os.path.isdir(gt_tp_dir):
        for f in sorted(os.listdir(gt_tp_dir)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(gt_tp_dir, f)
        print(f'GT masks loaded from MASK/Tp/: {len(gt_map)}')
    else:
        # Flat directory
        for f in sorted(os.listdir(GT_DIR)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(GT_DIR, f)
        print(f'GT masks loaded (flat): {len(gt_map)}')

# Match tampered images to GT masks
tp_with_gt = 0
tp_without_gt = 0
for tp in tp_paths:
    stem = Path(tp).stem.lower()
    # Try exact match and common variants
    variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
    found = any(v in gt_map for v in variants)
    if found:
        tp_with_gt += 1
    else:
        tp_without_gt += 1

USE_GT_MASKS = GT_DIR is not None and tp_with_gt > len(tp_paths) * 0.5

print(f'\nAuthentic images:  {len(au_paths)}')
print(f'Tampered images:   {len(tp_paths)}')
print(f'Total:             {len(au_paths) + len(tp_paths)}')
print(f'Class ratio (Au:Tp): {len(au_paths)/len(tp_paths):.2f}:1')
if GT_DIR:
    print(f'\nTampered with GT mask:    {tp_with_gt}')
    print(f'Tampered without GT mask: {tp_without_gt}')
print(f'\nUsing GT masks: {USE_GT_MASKS}')
if not USE_GT_MASKS:
    print('  -> Will generate pseudo-masks from ELA thresholding')


In [ ]:
# ============================================================
# 2.3 ELA Pseudo-Mask Generation (Fallback)
# ============================================================

def compute_ela(image_path, quality=90):
    """Compute Error Level Analysis map."""
    original = Image.open(image_path).convert('RGB')
    buffer = BytesIO()
    original.save(buffer, 'JPEG', quality=quality)
    buffer.seek(0)
    resaved = Image.open(buffer)
    ela = ImageChops.difference(original, resaved)
    extrema = ela.getextrema()
    max_diff = max(val[1] for val in extrema)
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff
    ela = ImageEnhance.Brightness(ela).enhance(scale)
    return ela

def generate_pseudo_mask(image_path, quality=90, threshold=50):
    """Generate a binary pseudo-mask from ELA.
    Pixels with ELA brightness above threshold are marked as tampered.
    """
    ela = compute_ela(image_path, quality)
    ela_gray = np.array(ela.convert('L'))
    # Adaptive threshold: use mean + 2*std of the ELA map
    mean_val = ela_gray.mean()
    std_val = ela_gray.std()
    adaptive_thresh = max(threshold, mean_val + 2 * std_val)
    mask = (ela_gray > adaptive_thresh).astype(np.float32)
    return mask

def get_gt_mask(image_path, target_size):
    """Get ground truth mask for an image.
    - Authentic images: all-zero mask
    - Tampered images: GT mask if available, else ELA pseudo-mask
    """
    is_tampered = '/Tp/' in image_path or '\\Tp\\' in image_path

    if not is_tampered:
        # Authentic â€” all zeros (no tampering)
        return np.zeros((target_size, target_size), dtype=np.float32)

    if USE_GT_MASKS:
        # Try to find matching GT mask
        stem = Path(image_path).stem.lower()
        variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
        for v in variants:
            if v in gt_map:
                mask = Image.open(gt_map[v]).convert('L')
                mask = mask.resize((target_size, target_size), Image.NEAREST)
                mask_arr = np.array(mask, dtype=np.float32)
                # Normalize to [0, 1]
                if mask_arr.max() > 1:
                    mask_arr = mask_arr / 255.0
                # Binarize
                mask_arr = (mask_arr > 0.5).astype(np.float32)
                return mask_arr

    # Fallback: ELA pseudo-mask
    try:
        mask = generate_pseudo_mask(image_path)
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_pil = mask_pil.resize((target_size, target_size), Image.NEAREST)
        return np.array(mask_pil, dtype=np.float32) / 255.0
    except Exception:
        return np.zeros((target_size, target_size), dtype=np.float32)

# Quick test
test_au = au_paths[0]
test_tp = tp_paths[0]
mask_au = get_gt_mask(test_au, IMAGE_SIZE)
mask_tp = get_gt_mask(test_tp, IMAGE_SIZE)
print(f'Authentic mask â€” shape: {mask_au.shape}, sum: {mask_au.sum():.0f} (should be 0)')
print(f'Tampered mask  â€” shape: {mask_tp.shape}, sum: {mask_tp.sum():.0f} (should be > 0)')

---

## 3. Data Preparation

### Input Pipeline — ELA + DCT Fusion (6 channels)

**ELA channels (0-2):** Standard ELA at Q=90
- Channel 0-2: RGB ELA map (pixel-level recompression artifacts)

**DCT channels (3-5):** Block-level frequency statistics
- Channel 3: AC energy (compression artifact strength per 8x8 block)
- Channel 4: DC coefficient (block mean luminance)
- Channel 5: HF energy (high-frequency detail/noise per block)

### Fusion Strategy

| Channel | Source | Domain | Resolution | Signal Type |
|---------|--------|--------|------------|-------------|
| 0-2 | ELA (Q=90) | Spatial (pixel) | Native 384x384 | Recompression residuals |
| 3-5 | DCT map | Frequency (block) | 48x48 → 384x384 | Coefficient statistics |

### Why Fusion?

ELA and DCT capture complementary tampering signals:
- **ELA** highlights pixel-level recompression artifacts — strong at fine boundaries
- **DCT** captures block-level compression statistics — strong at detecting double compression

Fusing both gives the model two independent views of the same evidence.

### Normalization
Each group is normalized independently:
- Channels 0-2: ELA mean/std (from training set)
- Channels 3-5: DCT mean/std (from training set)

### conv1 Modification
The pretrained conv1 expects 3 channels. For 6 channels:
1. Duplicate the pretrained 3-channel weights
2. Scale by 0.5 (preserve activation magnitude)
3. Unfreeze conv1 (must learn new channel fusion)

In [ ]:
# ============================================================
# 3.1 PyTorch Dataset and Transforms (CHANGED: ELA + DCT Fusion, 6ch)
# ============================================================

def compute_ela_image(image_path, quality=90, size=384):
    """Compute ELA as a 3-channel RGB image, resized to target size."""
    try:
        original = Image.open(image_path).convert('RGB')
        buffer = BytesIO()
        original.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        resaved = Image.open(buffer)
        ela = ImageChops.difference(original, resaved)
        extrema = ela.getextrema()
        max_diff = max(val[1] for val in extrema)
        if max_diff == 0:
            max_diff = 1
        scale = 255.0 / max_diff
        ela = ImageEnhance.Brightness(ela).enhance(scale)
        return np.array(ela.resize((size, size), Image.BILINEAR))  # (H, W, 3) uint8
    except Exception:
        return np.zeros((size, size, 3), dtype=np.uint8)


def compute_dct_feature_map(image_path, block_size=8, target_size=384):
    """Compute 3-channel DCT spatial feature map from an image.

    For each 8x8 block of the luminance channel:
      - Ch0: AC energy (sum of squared AC coefficients)
      - Ch1: DC coefficient (the [0,0] value)
      - Ch2: HF energy (sum of squares in bottom-right 4x4 quadrant)

    Returns: numpy array of shape (target_size, target_size, 3), dtype uint8
    """
    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if img is None:
            return np.zeros((target_size, target_size, 3), dtype=np.uint8)

        img_ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
        y_channel = img_ycrcb[:, :, 0].astype(np.float32)

        h, w = y_channel.shape
        h_aligned = h - (h % block_size)
        w_aligned = w - (w % block_size)
        y_channel = y_channel[:h_aligned, :w_aligned]

        n_blocks_h = h_aligned // block_size
        n_blocks_w = w_aligned // block_size

        if n_blocks_h == 0 or n_blocks_w == 0:
            return np.zeros((target_size, target_size, 3), dtype=np.uint8)

        ac_energy = np.zeros((n_blocks_h, n_blocks_w), dtype=np.float32)
        dc_value = np.zeros((n_blocks_h, n_blocks_w), dtype=np.float32)
        hf_energy = np.zeros((n_blocks_h, n_blocks_w), dtype=np.float32)

        for i in range(n_blocks_h):
            for j in range(n_blocks_w):
                block = y_channel[i*block_size:(i+1)*block_size,
                                  j*block_size:(j+1)*block_size]
                dct_block = cv2.dct(block)

                dc_value[i, j] = dct_block[0, 0]
                ac = dct_block.copy()
                ac[0, 0] = 0
                ac_energy[i, j] = np.sum(ac ** 2)
                hf_block = dct_block[4:, 4:]
                hf_energy[i, j] = np.sum(hf_block ** 2)

        def normalize_channel(ch):
            ch_min, ch_max = ch.min(), ch.max()
            if ch_max - ch_min < 1e-7:
                return np.zeros_like(ch, dtype=np.uint8)
            return ((ch - ch_min) / (ch_max - ch_min) * 255).astype(np.uint8)

        ac_norm = normalize_channel(ac_energy)
        dc_norm = normalize_channel(dc_value)
        hf_norm = normalize_channel(hf_energy)

        dct_map = np.stack([ac_norm, dc_norm, hf_norm], axis=-1)
        dct_resized = cv2.resize(dct_map, (target_size, target_size),
                                  interpolation=cv2.INTER_LINEAR)
        return dct_resized
    except Exception:
        return np.zeros((target_size, target_size, 3), dtype=np.uint8)


def compute_ela_statistics(image_paths, n_samples=500, size=384, quality=90):
    """Compute per-channel mean and std of ELA maps."""
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(len(image_paths), min(n_samples, len(image_paths)), replace=False)
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    n_pixels = 0
    for idx in tqdm(sample_idx, desc='ELA stats', leave=False):
        ela = compute_ela_image(image_paths[idx], quality=quality, size=size)
        arr = ela.astype(np.float64) / 255.0
        channel_sum += arr.reshape(-1, 3).sum(axis=0)
        channel_sq_sum += (arr.reshape(-1, 3) ** 2).sum(axis=0)
        n_pixels += arr.shape[0] * arr.shape[1]
    mean = channel_sum / n_pixels
    std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)
    std = np.maximum(std, 1e-5)
    return mean.tolist(), std.tolist()


def compute_dct_statistics(image_paths, n_samples=500, target_size=384):
    """Compute per-channel mean and std of DCT feature maps."""
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(len(image_paths), min(n_samples, len(image_paths)), replace=False)
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    n_pixels = 0
    for idx in tqdm(sample_idx, desc='DCT stats', leave=False):
        dct_map = compute_dct_feature_map(image_paths[idx], target_size=target_size)
        arr = dct_map.astype(np.float64) / 255.0
        channel_sum += arr.reshape(-1, 3).sum(axis=0)
        channel_sq_sum += (arr.reshape(-1, 3) ** 2).sum(axis=0)
        n_pixels += arr.shape[0] * arr.shape[1]
    mean = channel_sum / n_pixels
    std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)
    std = np.maximum(std, 1e-5)
    return mean.tolist(), std.tolist()

print('ELA + DCT feature extraction functions ready.')

# Placeholders --- will be set after split
ELA_MEAN = [0.5, 0.5, 0.5]
ELA_STD = [0.25, 0.25, 0.25]
DCT_MEAN = [0.5, 0.5, 0.5]
DCT_STD = [0.25, 0.25, 0.25]

class CASIAFusionDataset(Dataset):
    """CASIA v2.0 dataset --- 6-channel ELA + DCT fusion input."""

    def __init__(self, image_paths, labels, ela_mean, ela_std, dct_mean, dct_std,
                 mask_size=384, ela_quality=90):
        self.image_paths = image_paths
        self.labels = labels
        self.mask_size = mask_size
        self.ela_quality = ela_quality
        self.to_tensor = transforms.ToTensor()
        self.ela_normalize = transforms.Normalize(mean=ela_mean, std=ela_std)
        self.dct_normalize = transforms.Normalize(mean=dct_mean, std=dct_std)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]

        # ELA channels (3)
        ela = compute_ela_image(path, quality=self.ela_quality, size=self.mask_size)
        ela_tensor = self.to_tensor(ela)  # (3, H, W), [0, 1]
        ela_tensor = self.ela_normalize(ela_tensor)

        # DCT channels (3)
        dct_map = compute_dct_feature_map(path, target_size=self.mask_size)
        dct_tensor = self.to_tensor(dct_map)  # (3, H, W), [0, 1]
        dct_tensor = self.dct_normalize(dct_tensor)

        # Fuse: (6, H, W)
        fused = torch.cat([ela_tensor, dct_tensor], dim=0)

        mask = get_gt_mask(path, self.mask_size)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return fused, mask, label

print(f'Fusion dataset class ready (ELA Q={ELA_QUALITY} + DCT, 6ch total).')
print(f'Stats will be computed after data splitting.')

In [ ]:
# ============================================================
# 3.2 Data Splitting (70/15/15 Stratified) + ELA & DCT Stats
# ============================================================

all_paths = au_paths + tp_paths
all_labels = [0] * len(au_paths) + [1] * len(tp_paths)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED)

# Compute ELA normalization statistics
ELA_MEAN, ELA_STD = compute_ela_statistics(
    train_paths, n_samples=500, size=IMAGE_SIZE, quality=ELA_QUALITY)
print(f'ELA normalization statistics (from 500 training samples):')
print(f'  R Mean/Std: {ELA_MEAN[0]:.4f} / {ELA_STD[0]:.4f}')
print(f'  G Mean/Std: {ELA_MEAN[1]:.4f} / {ELA_STD[1]:.4f}')
print(f'  B Mean/Std: {ELA_MEAN[2]:.4f} / {ELA_STD[2]:.4f}')

# Compute DCT normalization statistics
DCT_MEAN, DCT_STD = compute_dct_statistics(
    train_paths, n_samples=500, target_size=IMAGE_SIZE)
print(f'\nDCT normalization statistics (from 500 training samples):')
print(f'  AC energy  Mean/Std: {DCT_MEAN[0]:.4f} / {DCT_STD[0]:.4f}')
print(f'  DC value   Mean/Std: {DCT_MEAN[1]:.4f} / {DCT_STD[1]:.4f}')
print(f'  HF energy  Mean/Std: {DCT_MEAN[2]:.4f} / {DCT_STD[2]:.4f}')

train_dataset = CASIAFusionDataset(
    train_paths, train_labels,
    ela_mean=ELA_MEAN, ela_std=ELA_STD,
    dct_mean=DCT_MEAN, dct_std=DCT_STD,
    mask_size=IMAGE_SIZE, ela_quality=ELA_QUALITY)
val_dataset = CASIAFusionDataset(
    val_paths, val_labels,
    ela_mean=ELA_MEAN, ela_std=ELA_STD,
    dct_mean=DCT_MEAN, dct_std=DCT_STD,
    mask_size=IMAGE_SIZE, ela_quality=ELA_QUALITY)
test_dataset = CASIAFusionDataset(
    test_paths, test_labels,
    ela_mean=ELA_MEAN, ela_std=ELA_STD,
    dct_mean=DCT_MEAN, dct_std=DCT_STD,
    mask_size=IMAGE_SIZE, ela_quality=ELA_QUALITY)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         persistent_workers=NUM_WORKERS > 0)

print(f'\nTrain: {len(train_dataset):>6} images  (Au: {sum(1 for l in train_labels if l==0)}, Tp: {sum(1 for l in train_labels if l==1)})')
print(f'Val:   {len(val_dataset):>6} images  (Au: {sum(1 for l in val_labels if l==0)}, Tp: {sum(1 for l in val_labels if l==1)})')
print(f'Test:  {len(test_dataset):>6} images  (Au: {sum(1 for l in test_labels if l==0)}, Tp: {sum(1 for l in test_labels if l==1)})')
print(f'\nTrain batches: {len(train_loader)}  (drop_last=True)')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')
print(f'Workers:       {NUM_WORKERS}  (persistent={NUM_WORKERS > 0})')

In [ ]:
# ============================================================
# 3.3 Sample Visualization (ELA + DCT Fusion Input)
# ============================================================

def denormalize_ela(tensor, mean=ELA_MEAN, std=ELA_STD):
    """Reverse ELA normalization for display."""
    t = tensor.clone()
    for ch in range(3):
        t[ch] = t[ch] * std[ch] + mean[ch]
    return t.clamp(0, 1)

def denormalize_dct(tensor, mean=DCT_MEAN, std=DCT_STD):
    """Reverse DCT normalization for display."""
    t = tensor.clone()
    for ch in range(3):
        t[ch] = t[ch] * std[ch] + mean[ch]
    return t.clamp(0, 1)

fig, axes = plt.subplots(4, 6, figsize=(24, 16))

sample_indices = []
au_shown, tp_shown = 0, 0
for i in range(len(train_dataset)):
    if train_labels[i] == 0 and au_shown < 2:
        sample_indices.append(i)
        au_shown += 1
    elif train_labels[i] == 1 and tp_shown < 2:
        sample_indices.append(i)
        tp_shown += 1
    if au_shown >= 2 and tp_shown >= 2:
        break

for row, idx in enumerate(sample_indices):
    fused_tensor, mask, label = train_dataset[idx]

    # Split fused tensor: ELA (ch 0-2), DCT (ch 3-5)
    ela_part = denormalize_ela(fused_tensor[:3])
    dct_part = denormalize_dct(fused_tensor[3:])

    # Column 0: ELA RGB
    ela_display = ela_part.permute(1, 2, 0).numpy()
    axes[row, 0].imshow(ela_display)
    axes[row, 0].set_title(f'ELA ({"Au" if label==0 else "Tp"})', fontsize=10)
    axes[row, 0].axis('off')

    # Column 1: DCT Combined
    dct_display = dct_part.permute(1, 2, 0).numpy()
    axes[row, 1].imshow(dct_display)
    axes[row, 1].set_title('DCT Combined', fontsize=10)
    axes[row, 1].axis('off')

    # Column 2: DCT AC Energy
    axes[row, 2].imshow(dct_part[0].numpy(), cmap='viridis', vmin=0, vmax=1)
    axes[row, 2].set_title('DCT AC Energy', fontsize=10)
    axes[row, 2].axis('off')

    # Column 3: DCT HF Energy
    axes[row, 3].imshow(dct_part[2].numpy(), cmap='viridis', vmin=0, vmax=1)
    axes[row, 3].set_title('DCT HF Energy', fontsize=10)
    axes[row, 3].axis('off')

    # Column 4: GT Mask
    mask_display = mask.squeeze(0).numpy()
    axes[row, 4].imshow(mask_display, cmap='hot', vmin=0, vmax=1)
    axes[row, 4].set_title(f'GT Mask (sum={mask_display.sum():.0f})', fontsize=10)
    axes[row, 4].axis('off')

    # Column 5: ELA + Mask overlay
    overlay = ela_display.copy()
    mask_rgb = np.zeros_like(overlay)
    mask_rgb[:, :, 0] = mask_display
    overlay = overlay * 0.7 + mask_rgb * 0.3
    overlay = np.clip(overlay, 0, 1)
    axes[row, 5].imshow(overlay)
    axes[row, 5].set_title('ELA + Mask Overlay', fontsize=10)
    axes[row, 5].axis('off')

plt.suptitle('Fusion Input: ELA | DCT Combined | AC Energy | HF Energy | GT Mask | Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

## 4. Model Architecture

### UNet with ResNet-34 Encoder — 6-Channel Fusion Input

```
Input: 384x384x6 (ch0-2: ELA Q=90, ch3-5: DCT spatial map)
|
+-- ENCODER (ResNet-34, ImageNet pretrained)
|   +-- conv1:  7x7, 64, stride 2   -> 192x192x64    [UNFROZEN for 6ch, BN UNFROZEN]
|   +-- pool:   3x3, stride 2       -> 96x96x64       [FROZEN]
|   +-- layer1: 3x[3x3, 64]         -> 96x96x64       [FROZEN, BN UNFROZEN]  [skip 1]
|   +-- layer2: 4x[3x3, 128]        -> 48x48x128      [FROZEN, BN UNFROZEN]  [skip 2]
|   +-- layer3: 6x[3x3, 256]        -> 24x24x256      [FROZEN, BN UNFROZEN]  [skip 3]
|   +-- layer4: 3x[3x3, 512]        -> 12x12x512      [FROZEN, BN UNFROZEN]  [skip 4]
|
+-- DECODER (UNet, TRAINABLE, lr=1e-3, ~500K params)
|   +-- up1: 12->24,  concat skip 3  -> 24x24
|   +-- up2: 24->48,  concat skip 2  -> 48x48
|   +-- up3: 48->96,  concat skip 1  -> 96x96
|   +-- up4: 96->192, concat skip 0  -> 192x192
|   +-- final: -> 384x384, 1x1 conv  -> sigmoid
|
Output: 384x384x1 (pixel probability)
```

### Key Design Decisions

| Decision | Choice | Reason |
|----------|--------|--------|
| Input | **6ch ELA+DCT** | Dual-domain fusion: spatial + frequency |
| Ch 0-2 | ELA (Q=90, RGB) | Pixel-level recompression artifacts |
| Ch 3-5 | DCT (AC/DC/HF) | Block-level compression statistics |
| conv1 | **UNFROZEN** | Must learn to combine 6 channels (pretrained weights duplicated, scaled 0.5) |
| Encoder body | **FROZEN** | Conv weights extract spatial features |
| Encoder BN | **UNFROZEN** | Adapts to new input distribution |

### conv1 Initialization
Pretrained conv1 has shape (64, 3, 7, 7). For 6 channels:
- Duplicate: `cat([w3ch, w3ch.clone()], dim=1)` → (64, 6, 7, 7)
- Scale by 0.5 to preserve activation magnitude
- Unfreeze to allow learning new channel fusion

In [ ]:
# ============================================================
# 4.1 Build Model (6ch conv1, Frozen Body + BN Unfrozen)
# ============================================================

# Step 1: Build a temporary 3ch model to get pretrained conv1 weights
pretrained_3ch = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
)
pretrained_conv1_weight = pretrained_3ch.encoder.conv1.weight.data.clone()  # (64, 3, 7, 7)

# Step 2: Build the actual 6ch model
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=IN_CHANNELS,  # 6
    classes=NUM_CLASSES,
    activation=None
)

# Step 3: Initialize 6ch conv1 from pretrained 3ch weights
# Duplicate and scale by 0.5 to preserve activation magnitude
model.encoder.conv1.weight.data = torch.cat([
    pretrained_conv1_weight,
    pretrained_conv1_weight.clone()
], dim=1) / 2.0  # (64, 6, 7, 7)

print(f'conv1 initialized: pretrained 3ch weights duplicated and scaled by 0.5')
print(f'  conv1 weight shape: {model.encoder.conv1.weight.shape}')

del pretrained_3ch  # free memory

# Step 4: Freeze ALL encoder parameters
for param in model.encoder.parameters():
    param.requires_grad = False

# Step 5: Unfreeze conv1 (must learn 6-channel fusion)
for param in model.encoder.conv1.parameters():
    param.requires_grad = True

# Step 6: Unfreeze ONLY BatchNorm layers in encoder (domain adaptation)
bn_param_count = 0
for module in model.encoder.modules():
    if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
        for param in module.parameters():
            param.requires_grad = True
            bn_param_count += param.numel()
        module.track_running_stats = True

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
conv1_params = sum(p.numel() for p in model.encoder.conv1.parameters())
decoder_trainable = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
seghead_trainable = sum(p.numel() for p in model.segmentation_head.parameters() if p.requires_grad)

print(f'\nModel: UNet + {ENCODER} ({ENCODER_WEIGHTS}) — 6ch FUSION, conv1 UNFROZEN')
print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'  conv1 params:       {conv1_params:>12,}  (UNFROZEN, lr={LEARNING_RATE})')
print(f'  Encoder BN params:  {bn_param_count:>12,}  (BatchNorm only)')
print(f'  Decoder:            {decoder_trainable:>12,}  (lr={LEARNING_RATE})')
print(f'  Segmentation head:  {seghead_trainable:>12,}  (lr={LEARNING_RATE})')
print(f'Frozen parameters:    {frozen_params:>12,}  (conv weights in layer1-4)')
print(f'Trainable ratio:      {trainable_params/total_params*100:.1f}%')
print(f'Data:param ratio:     1 : {trainable_params/len(train_dataset):.0f}')

---

## 5. Training

### Configuration

| Parameter | Value |
|-----------|-------|
| Loss | BCE + Dice (combined) |
| Optimizer | Adam (single LR) |
| LR | **1e-3** (decoder + encoder BN) |
| Scheduler | ReduceLROnPlateau (factor=0.5, patience=3) |
| Early stopping | patience=7, monitor=val_loss |
| Epochs | 25 max |
| Batch size | 16 |

### Why Single LR (Not Differential)?

In vR.P.2, differential LR was needed because conv weights in layer3+4 were unfrozen.
In vR.P.3, all conv weights are frozen. Only BatchNorm params and decoder are trainable.
BN params (gamma, beta) and decoder params can safely share the same LR as they are
all learning to adapt to the new ELA domain.

In [ ]:
# ============================================================
# 5.1 Loss, Optimizer, Scheduler
# ============================================================

bce_loss_fn = smp.losses.SoftBCEWithLogitsLoss()
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)

def criterion(pred, target):
    return bce_loss_fn(pred, target) + dice_loss_fn(pred, target)

# Single optimizer â€” all trainable params at same LR (vR.P.3)
trainable_params_list = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params_list, lr=LEARNING_RATE, weight_decay=1e-5)

print(f'Optimizer: Adam (single LR)')
print(f'  Trainable params: {sum(p.numel() for p in trainable_params_list):,}')
print(f'  LR: {LEARNING_RATE}')

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ============================================================
# 5.2 Training and Validation Functions (AMP-enabled)
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0
    num_batches = 0
    for images, masks, labels in tqdm(loader, desc='Train', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            predictions = model(images)
            loss = criterion(predictions, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    num_batches = 0
    all_preds = []
    all_masks = []
    for images, masks, labels in tqdm(loader, desc='Val', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with autocast('cuda'):
            predictions = model(images)
            loss = criterion(predictions, masks)
        total_loss += loss.item()
        num_batches += 1
        probs = torch.sigmoid(predictions.float())
        all_preds.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
    avg_loss = total_loss / num_batches
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    binary_preds = (all_preds > 0.5).astype(np.float32)
    pred_flat = binary_preds.flatten()
    mask_flat = all_masks.flatten()
    eps = 1e-7
    tp = (pred_flat * mask_flat).sum()
    fp = (pred_flat * (1 - mask_flat)).sum()
    fn = ((1 - pred_flat) * mask_flat).sum()
    pixel_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    pixel_iou = tp / (tp + fp + fn + eps)
    return avg_loss, pixel_f1, pixel_iou

print('Training functions ready (AMP-enabled).')

In [ ]:
# ============================================================
# 5.3 Training Loop (with checkpoint save/resume + AMP)
# ============================================================

scaler = GradScaler('cuda')

history = {
    'train_loss': [], 'val_loss': [], 'val_pixel_f1': [], 'val_pixel_iou': [],
    'lr': []
}

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
best_model_state = None
start_epoch = 1

if os.path.exists(CHECKPOINT_PATH):
    print(f'Checkpoint found: {CHECKPOINT_PATH}')
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    history = ckpt['history']
    if ckpt.get('best_model_state') is not None:
        best_model_state = ckpt['best_model_state']
    print(f'  Resuming from epoch {start_epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})')
else:
    print('No checkpoint found \u2014 starting fresh.')

print(f'Starting training: epochs {start_epoch}-{EPOCHS}, patience={PATIENCE}')
print(f'LR: {LEARNING_RATE} | Input: ELA+DCT fusion ({IN_CHANNELS}ch) | AMP: Enabled')
print(f'{"="*80}')

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_f1, val_iou = validate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pixel_f1'].append(val_f1)
    history['val_pixel_iou'].append(val_iou)
    history['lr'].append(current_lr)
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        improved = ' *'
    else:
        patience_counter += 1
    print(f'Epoch {epoch:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Pixel F1: {val_f1:.4f} | IoU: {val_iou:.4f} | LR: {current_lr:.2e}{improved}')
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss, 'best_epoch': best_epoch,
        'patience_counter': patience_counter, 'best_model_state': best_model_state,
        'history': history,
    }, CHECKPOINT_PATH)
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}')
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model = model.to(DEVICE)
    print(f'\nRestored best model from epoch {best_epoch} (val_loss={best_val_loss:.4f})')
else:
    print('\nNo improvement during training \u2014 using final weights')

print(f'{"="*80}')
print(f'Training complete. Best epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}')

---

## 6. Evaluation

### Metrics Computed

**Pixel-Level (Localization):**
- Pixel F1 Score (= Dice coefficient)
- IoU (Intersection over Union / Jaccard index)
- Pixel AUC (ROC-AUC on per-pixel probabilities)

**Image-Level (Classification):**
- Accuracy (image classified as tampered if mask has any positive pixel above threshold)
- Per-class Precision, Recall, F1
- Macro F1
- ROC-AUC
- Confusion Matrix

In [ ]:
# ============================================================
# 6.1 Test Set Evaluation â€” Pixel-Level Metrics
# ============================================================

@torch.no_grad()
def evaluate_test(model, loader, device):
    """Full evaluation on test set."""
    model.eval()
    all_probs = []
    all_masks = []
    all_labels = []

    for images, masks, labels in tqdm(loader, desc='Test Eval'):
        images = images.to(device)
        predictions = model(images)
        probs = torch.sigmoid(predictions)

        all_probs.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_labels.extend(labels.numpy())

    all_probs = np.concatenate(all_probs, axis=0)  # (N, 1, H, W)
    all_masks = np.concatenate(all_masks, axis=0)   # (N, 1, H, W)
    all_labels = np.array(all_labels)

    return all_probs, all_masks, all_labels

test_probs, test_masks, test_labels = evaluate_test(model, test_loader, DEVICE)
test_preds_binary = (test_probs > 0.5).astype(np.float32)

# Pixel-level metrics (flatten all pixels)
pred_flat = test_preds_binary.flatten()
mask_flat = test_masks.flatten()
prob_flat = test_probs.flatten()

eps = 1e-7
tp = (pred_flat * mask_flat).sum()
fp = (pred_flat * (1 - mask_flat)).sum()
fn = ((1 - pred_flat) * mask_flat).sum()
tn = ((1 - pred_flat) * (1 - mask_flat)).sum()

pixel_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
pixel_iou = tp / (tp + fp + fn + eps)
pixel_dice = pixel_f1  # Dice = F1 for binary
pixel_precision = tp / (tp + fp + eps)
pixel_recall = tp / (tp + fn + eps)

# Pixel AUC (subsample for speed if needed)
n_pixels = len(prob_flat)
if n_pixels > 5_000_000:
    sample_idx = np.random.choice(n_pixels, 5_000_000, replace=False)
    pixel_auc = roc_auc_score(mask_flat[sample_idx], prob_flat[sample_idx])
else:
    pixel_auc = roc_auc_score(mask_flat, prob_flat) if mask_flat.sum() > 0 and (1-mask_flat).sum() > 0 else 0.0

print(f'{"="*60}')
print(f'  PIXEL-LEVEL METRICS (Test Set)')
print(f'{"="*60}')
print(f'  Pixel Precision:  {pixel_precision:.4f}')
print(f'  Pixel Recall:     {pixel_recall:.4f}')
print(f'  Pixel F1 (Dice):  {pixel_f1:.4f}')
print(f'  Pixel IoU:        {pixel_iou:.4f}')
print(f'  Pixel AUC:        {pixel_auc:.4f}')
print(f'{"="*60}')

In [ ]:
# ============================================================
# 6.2 Test Set Evaluation â€” Image-Level Classification
# ============================================================

# Derive image-level classification from masks:
# An image is classified as "tampered" if any predicted pixel > threshold
MASK_AREA_THRESHOLD = 100  # minimum number of tampered pixels to classify as tampered

image_pred_labels = []
image_pred_scores = []

for i in range(len(test_probs)):
    prob_map = test_probs[i, 0]  # (H, W)
    binary_map = (prob_map > 0.5).astype(np.float32)
    tampered_pixel_count = binary_map.sum()

    # Classification: tampered if enough pixels predicted as tampered
    pred_label = 1 if tampered_pixel_count >= MASK_AREA_THRESHOLD else 0
    image_pred_labels.append(pred_label)

    # Score: max probability in the mask (for ROC-AUC)
    image_pred_scores.append(prob_map.max())

image_pred_labels = np.array(image_pred_labels)
image_pred_scores = np.array(image_pred_scores)

# Classification metrics
cls_accuracy = accuracy_score(test_labels, image_pred_labels)
cls_report = classification_report(test_labels, image_pred_labels,
                                     target_names=['Authentic', 'Tampered'],
                                     output_dict=True)
cls_macro_f1 = f1_score(test_labels, image_pred_labels, average='macro')
cls_auc = roc_auc_score(test_labels, image_pred_scores) if len(np.unique(test_labels)) > 1 else 0.0

print(f'{"="*60}')
print(f'  IMAGE-LEVEL CLASSIFICATION (Test Set)')
print(f'{"="*60}')
print(f'  Test Accuracy:    {cls_accuracy:.4f}  ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:         {cls_macro_f1:.4f}')
print(f'  ROC-AUC:          {cls_auc:.4f}')
print(f'')
print(f'  Per-Class Results:')
print(f'  {"":>12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
for cls_name in ['Authentic', 'Tampered']:
    r = cls_report[cls_name]
    print(f'  {cls_name:>12} {r["precision"]:>10.4f} {r["recall"]:>10.4f} {r["f1-score"]:>10.4f} {r["support"]:>10.0f}')
print(f'{"="*60}')

# Classification report (full)
print('\nFull Classification Report:')
print(classification_report(test_labels, image_pred_labels, target_names=['Authentic', 'Tampered']))

In [ ]:
# ============================================================
# 6.3 Confusion Matrix and ROC Curve
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(test_labels, image_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (Acc={cls_accuracy:.2%})')

# Print confusion details
tn, fp, fn, tp_cls = cm.ravel()
total = cm.sum()
print(f'Confusion Matrix:')
print(f'  TN={tn}, FP={fp}, FN={fn}, TP={tp_cls}')
print(f'  FP Rate: {fp/(tn+fp)*100:.1f}%')
print(f'  FN Rate: {fn/(fn+tp_cls)*100:.1f}%')

# ROC Curve
if len(np.unique(test_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(test_labels, image_pred_scores)
    axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {cls_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC={cls_auc:.4f})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Classification Performance', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.4 Training Curves
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
axes[0, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Pixel F1
axes[0, 1].plot(epochs_range, history['val_pixel_f1'], 'g-', linewidth=2)
axes[0, 1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Pixel F1')
axes[0, 1].set_title('Validation Pixel F1')
axes[0, 1].grid(True, alpha=0.3)

# Pixel IoU
axes[1, 0].plot(epochs_range, history['val_pixel_iou'], 'm-', linewidth=2)
axes[1, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Pixel IoU')
axes[1, 0].set_title('Validation Pixel IoU')
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs_range, history['lr'], 'k-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Training History', fontsize=14)
plt.tight_layout()
plt.show()

---

## 7. Visualization

### Original â†’ Ground Truth â†’ Predicted â†’ Overlay

The key deliverable: pixel-level tampered region predictions visualized alongside the original image and ground truth mask.

In [ ]:
# ============================================================
# 7.1 Original / Ground Truth / Predicted / Overlay Grid
# ============================================================

def visualize_predictions(model, dataset, indices, device, title='Predictions'):
    """Display ELA Input | GT Mask | Predicted Mask | Overlay for each index."""
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    model.eval()

    for row, idx in enumerate(indices):
        img_tensor, gt_mask, label = dataset[idx]

        # Predict
        with torch.no_grad():
            pred_logit = model(img_tensor.unsqueeze(0).to(device))
            pred_prob = torch.sigmoid(pred_logit).cpu().squeeze().numpy()

        pred_binary = (pred_prob > 0.5).astype(np.float32)

        # Denormalize ELA part (first 3 channels) for display
        ela_display = denormalize_ela(img_tensor[:3]).permute(1, 2, 0).numpy()
        gt_display = gt_mask.squeeze(0).numpy()

        # Col 0: ELA Input (first 3 channels of fusion)
        axes[row, 0].imshow(ela_display)
        axes[row, 0].set_title(f'ELA Input ({"Tampered" if label==1 else "Authentic"})', fontsize=11)
        axes[row, 0].axis('off')

        # Col 1: Ground Truth
        axes[row, 1].imshow(gt_display, cmap='hot', vmin=0, vmax=1)
        axes[row, 1].set_title(f'Ground Truth (sum={gt_display.sum():.0f})', fontsize=11)
        axes[row, 1].axis('off')

        # Col 2: Predicted Mask
        axes[row, 2].imshow(pred_binary, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title(f'Predicted (sum={pred_binary.sum():.0f})', fontsize=11)
        axes[row, 2].axis('off')

        # Col 3: Overlay
        overlay = ela_display.copy()
        overlay_mask = np.zeros_like(overlay)
        overlay_mask[:, :, 1] = gt_display * 0.4       # Green = GT
        overlay_mask[:, :, 0] = pred_binary * 0.4       # Red = Predicted
        combined = np.clip(overlay * 0.6 + overlay_mask, 0, 1)
        axes[row, 3].imshow(combined)
        axes[row, 3].set_title('Overlay (green=GT, red=pred)', fontsize=11)
        axes[row, 3].axis('off')

    plt.suptitle(f'{VERSION} — {title}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

# Select sample images
tampered_indices = [i for i, l in enumerate(test_labels) if l == 1]
authentic_indices = [i for i, l in enumerate(test_labels) if l == 0]

sample_tp = tampered_indices[:4]
sample_au = authentic_indices[:2]

print('--- Tampered Image Predictions ---')
visualize_predictions(model, test_dataset, sample_tp, DEVICE, 'Tampered Images')

print('\n--- Authentic Image Predictions ---')
visualize_predictions(model, test_dataset, sample_au, DEVICE, 'Authentic Images')

In [ ]:
# ============================================================
# 7.2 Per-Image Metric Distribution
# ============================================================

# Compute per-image pixel F1
per_image_f1 = []
per_image_iou = []
per_image_labels = []

for i in range(len(test_probs)):
    pred = (test_probs[i].flatten() > 0.5).astype(np.float32)
    mask = test_masks[i].flatten()

    tp_i = (pred * mask).sum()
    fp_i = (pred * (1 - mask)).sum()
    fn_i = ((1 - pred) * mask).sum()

    f1_i = (2 * tp_i) / (2 * tp_i + fp_i + fn_i + 1e-7)
    iou_i = tp_i / (tp_i + fp_i + fn_i + 1e-7)

    per_image_f1.append(f1_i)
    per_image_iou.append(iou_i)
    per_image_labels.append(test_labels[i])

per_image_f1 = np.array(per_image_f1)
per_image_iou = np.array(per_image_iou)
per_image_labels = np.array(per_image_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[0].hist(per_image_f1[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[0].set_xlabel('Pixel F1 Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-Image Pixel F1 Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[1].hist(per_image_iou[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[1].set_xlabel('Pixel IoU Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-Image Pixel IoU Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Per-Image Metric Distributions', fontsize=14)
plt.tight_layout()
plt.show()

# Summary stats
print(f'Per-Image Pixel F1:')
print(f'  Tampered:  mean={per_image_f1[per_image_labels==1].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==1].std():.4f}')
print(f'  Authentic: mean={per_image_f1[per_image_labels==0].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==0].std():.4f}')

---

## 8. Results Summary

In [ ]:
# ============================================================
# 8.1 Results Tracking Table
# ============================================================

print(f'{"="*100}')
print(f'  RESULTS SUMMARY --- {VERSION}')
print(f'{"="*100}')
print()

# Ablation comparison table
print('Cross-Track Comparison:')
print(f'{"Version":<10} {"Track":<12} {"Encoder":<12} {"Input":<16} '
      f'{"Test Acc":<10} {"Pixel-F1":<10} {"IoU":<8}')
print('-' * 86)

print(f'{"vR.P.3":<10} {"Pretrained":<12} {"ResNet-34":<12} {"ELA Q90 384sq":<16} '
      f'{"86.79%":<10} {"0.6920":<10} {"0.5291":<8}')

print(f'{"vR.P.10":<10} {"Pretrained":<12} {"ResNet-34":<12} {"ELA Q90 384sq":<16} '
      f'{"87.32%":<10} {"0.7277":<10} {"0.5719":<8}')

# This run
print(f'{VERSION:<10} {"Pretrained":<12} {"ResNet-34":<12} {"ELA+DCT 384sq":<16} '
      f'{cls_accuracy*100:.2f}{"%-":<7} {pixel_f1:<10.4f} {pixel_iou:<8.4f}')
print()

# Full metrics
print(f'Pixel-Level Metrics:')
print(f'  Pixel F1 (Dice): {pixel_f1:.4f}')
print(f'  Pixel IoU:       {pixel_iou:.4f}')
print(f'  Pixel Precision: {pixel_precision:.4f}')
print(f'  Pixel Recall:    {pixel_recall:.4f}')
print(f'  Pixel AUC:       {pixel_auc:.4f}')
print()
print(f'Image-Level Metrics:')
print(f'  Accuracy:        {cls_accuracy:.4f} ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:        {cls_macro_f1:.4f}')
print(f'  ROC-AUC:         {cls_auc:.4f}')
print(f'  Au Precision:    {cls_report["Authentic"]["precision"]:.4f}')
print(f'  Au Recall:       {cls_report["Authentic"]["recall"]:.4f}')
print(f'  Au F1:           {cls_report["Authentic"]["f1-score"]:.4f}')
print(f'  Tp Precision:    {cls_report["Tampered"]["precision"]:.4f}')
print(f'  Tp Recall:       {cls_report["Tampered"]["recall"]:.4f}')
print(f'  Tp F1:           {cls_report["Tampered"]["f1-score"]:.4f}')
print()
print(f'Training:')
print(f'  Best epoch:      {best_epoch}')
print(f'  Epochs trained:  {len(history["train_loss"])}')
print(f'  Best val loss:   {best_val_loss:.4f}')
print(f'{"="*100}')

---

## 9. Discussion

### What Changed in vR.P.17

**ELA + DCT fusion input:** Instead of 3-channel ELA (P.3), this experiment concatenates 3 ELA channels with 3 DCT spatial map channels to create a 6-channel input. conv1 is modified and unfrozen to learn the new channel fusion.

### Fusion vs Single-Input Approaches

| Aspect | P.3 (ELA only) | P.16 (DCT only) | P.17 (ELA+DCT fusion) |
|--------|----------------|------------------|----------------------|
| Channels | 3 (ELA RGB) | 3 (AC/DC/HF) | 6 (ELA + DCT) |
| Spatial domain | Yes | No | Yes |
| Frequency domain | No | Yes | Yes |
| conv1 | Frozen (3ch pretrained) | Frozen (3ch pretrained) | Unfrozen (6ch modified) |
| Information diversity | Low (3 correlated RGB) | Medium (3 independent stats) | High (6 channels, 2 domains) |

### Historical Parallel

vR.P.4 (4-channel RGB+ELA) achieved Pixel F1 = 0.7053 vs P.3's 0.6920 (+0.0133). That 4-channel fusion was NEUTRAL. This 6-channel fusion (ELA+DCT) may follow a similar pattern — adding channels doesn't guarantee improvement when the primary signal (ELA) is already strong.

### Risk Factors
1. **6-channel conv1 instability**: Even with 0.5 scaling, the duplicated weights may destabilize early training
2. **DCT noise**: DCT channels may add noise rather than complementary signal (cf. P.4)
3. **Doubled preprocessing**: Both ELA and DCT computed per image increases training time

### Key Findings
*(To be filled after running on Kaggle)*

### Next Steps
- If positive (F1 > 0.7277): fusion approach validated; explore attention-based channel weighting
- If neutral: DCT adds marginal signal; ELA remains dominant
- If negative: fusion confuses the model; stick with single-input approaches

In [ ]:
# ============================================================
# 10. Save Model
# ============================================================

model_filename = f'{VERSION}_unet_resnet34_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'history': history,
    'config': {
        'version': VERSION, 'encoder': ENCODER, 'encoder_weights': ENCODER_WEIGHTS,
        'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'ela_quality': ELA_QUALITY,
        'dct_block_size': DCT_BLOCK_SIZE,
        'input_type': 'ELA+DCT', 'in_channels': IN_CHANNELS,
        'epochs_trained': len(history['train_loss']),
        'seed': SEED,
    }
}, model_filename)

print(f'Model saved: {model_filename}')
print(f'File size: {os.path.getsize(model_filename) / 1e6:.1f} MB')
print(f'\n{VERSION} complete.')